# Setup auxiliary disco2very activities

Creates only the custom helper activities required by `my_activities.py` for the cradle-to-gate ethylene routes. No LCA calculations, plotting, or exploratory tests are included.

In [6]:
import bw2data as bd
import pandas as pd

PROJECT = "optimex_remind"
ECOINVENT_DB = "ecoinvent-3.12-cutoff"
BIOSPHERE_DB = "ecoinvent-3.12-biosphere"
CUSTOM_DB = "disco2very"

bd.projects.set_current(PROJECT)
missing = [name for name in [ECOINVENT_DB, BIOSPHERE_DB] if name not in bd.databases]
if missing:
    raise RuntimeError(f"Missing required Brightway database(s) in project {PROJECT!r}: {missing}")

eidb = bd.Database(ECOINVENT_DB)
bsdb = bd.Database(BIOSPHERE_DB)
custom_db = bd.Database(CUSTOM_DB)
if CUSTOM_DB not in bd.databases:
    custom_db.register()

created = []

In [7]:
def recreate(code, *, name, location, unit, comment):
    try:
        activity = bd.get_activity((CUSTOM_DB, code))
        for exchange in list(activity.exchanges()):
            exchange.delete()
        activity["name"] = name
        activity["location"] = location
        activity["unit"] = unit
        activity["comment"] = comment
        activity.save()
        return activity
    except Exception:
        activity = custom_db.new_activity(code=code, name=name, location=location, unit=unit, comment=comment)
        activity.save()
        return activity


def production(activity, product, unit=None):
    activity.new_exchange(type="production", name=product, amount=1, unit=unit or activity["unit"], input=activity).save()
    activity["reference product"] = product
    activity.save()


def tech(activity, input_activity, amount):
    activity.new_exchange(
        type="technosphere",
        name=input_activity["name"],
        unit=input_activity["unit"],
        amount=amount,
        input=input_activity,
    ).save()


def bio(activity, input_flow, amount):
    activity.new_exchange(
        type="biosphere",
        name=input_flow["name"],
        unit=input_flow["unit"],
        amount=amount,
        input=input_flow,
    ).save()


def remember(activity, status="created"):
    created.append({
        "activity name": activity["name"],
        "code": activity["code"],
        "database": activity.key[0],
        "location": activity.get("location"),
        "unit": activity.get("unit"),
        "reference product": activity.get("reference product"),
        "modelling status": status,
    })

## Cooling energy helpers

In [9]:
coolmin25 = eidb.get(name="cooling energy production, at -25 °C, propylene compression refrigeration system 1 MW", location="GLO")
coolmin45 = eidb.get(name="cooling energy production, at -45 °C, propylene compression refrigeration system 1 MW", location="GLO")
coolmin30 = recreate(
    "cooling_minus_30_v1",
    name="cooling energy production, at -30 °C, propylene compression refrigeration system 1 MW",
    location="GLO",
    unit="megajoule",
    comment="Linear interpolation between -25 °C cooling (75%) and -45 °C cooling (25%).",
)
production(coolmin30, "cooling energy, at -30 °C", "megajoule")
tech(coolmin30, coolmin25, 0.75)
tech(coolmin30, coolmin45, 0.25)
remember(coolmin30)

coolmin100 = eidb.get(name="market for cooling energy, at -100 °C", location="GLO")
coolmin55 = eidb.get(name="market for cooling energy, at -55 °C", location="GLO")
coolmin75 = recreate(
    "cooling_minus_75_v1",
    name="cooling energy production, at -75 °C",
    location="GLO",
    unit="megajoule",
    comment="Linear interpolation between -100 °C cooling (4/9) and -55 °C cooling (5/9).",
)
production(coolmin75, "cooling energy, at -75 °C", "megajoule")
tech(coolmin75, coolmin100, 4 / 9)
tech(coolmin75, coolmin55, 5 / 9)
remember(coolmin75)

## PEM and DAC helper materials

In [10]:
co2_flow = bsdb.get("349b29d1-3e58-4c66-98b9-9d1a076efd2e")
so2_flow = bsdb.get("fd7aa71c-508c-480d-81a6-8052aad92646")
phosphorus_flow = bsdb.get("2d4b8ec1-8d53-4e62-8a11-ebc45909b02e")

iridium = recreate(
    "iridium production",
    name="iridium production",
    location="GLO",
    unit="kilogram",
    comment="Iridium production including mining, purification, and refining.",
)
production(iridium, "iridium", "kilogram")
bio(iridium, co2_flow, 8860)
bio(iridium, so2_flow, 3100 / 1.2)
bio(iridium, phosphorus_flow, 28 / 3.06)
remember(iridium)

In [11]:
ethanolamine = eidb.get(name="market for monoethanolamine", location="GLO")
sodium_sulfate = eidb.get(name="sodium sulfate production, from natural sources", location="RER")
sodium_hydroxide = eidb.get(name="market for sodium hydroxide, without water, in 50% solution state", location="RER")
hydrochloric_acid = eidb.get(name="market for hydrochloric acid, without water, in 30% solution state", location="RER")
ethanol = eidb.get(name="market for ethanol, without water, in 99.7% solution state, from ethylene", location="RER")
diethyl_ether = eidb.get(name="ethanol production, ethylene hydration", location="RER", product="diethyl ether, without water, in 99.95% solution state")
water_deionised = eidb.get(name="water production, deionised", location="Europe without Switzerland")
electricity = eidb.get(name="market for electricity, medium voltage", location="DE")
heat = eidb.get(name="heat production, natural gas, at industrial furnace low-NOx >100kW", location="Europe without Switzerland")
spent_solvent = eidb.get(name="market for spent solvent mixture", location="Europe without Switzerland")

pei = recreate(
    "polyethyleneimine production",
    name="polyethyleneimine production",
    location="DE",
    unit="kilogram",
    comment="Polyethyleneimine production based on Deutz and Bardow supplementary information (2021).",
)
production(pei, "polyethyleneimine", "kilogram")
for input_activity, amount in [
    (ethanolamine, 1.42),
    (sodium_sulfate, -0.86),
    (sodium_hydroxide, 3.72),
    (hydrochloric_acid, 0.170666667),
    (ethanol, 2.84),
    (diethyl_ether, 33.94),
    (water_deionised, 11.99),
    (electricity, 0.42),
    (heat, 2.35),
    (spent_solvent, -0.37),
]:
    tech(pei, input_activity, amount)
remember(pei)

## Direct air capture infrastructure and adsorbent

In [12]:
concrete_treat = eidb.get(name="treatment of waste concrete, inert material landfill", location="Europe without Switzerland")
steel_recycling = eidb.get(name="treatment of waste reinforcement steel, recycling", location="CH")
steel_disposal = eidb.get(name="treatment of waste reinforcement steel, collection for final disposal", location="CH")
wool_treat = eidb.get(name="treatment of waste mineral wool, inert material landfill", location="RoW")
pu_treat = eidb.get(name="treatment of waste polyurethane, municipal incineration", location="GLO", product="waste polyurethane")
copper_treat = eidb.get(name="treatment of waste copper, municipal incineration", location="Europe without Switzerland")
aluminium_treat = eidb.get(name="treatment of waste aluminium, sanitary landfill", location="CH", product="waste aluminium")
rubber_treat = eidb.get(name="treatment of waste rubber, unspecified, municipal incineration", location="Europe without Switzerland", product="waste rubber, unspecified")

dac_treatment = recreate(
    "treatment of direct air capture, 2016",
    name="treatment of direct air capture, 2016",
    location="RER",
    unit="unit",
    comment="End-of-life treatment of direct air capture infrastructure based on Deutz and Bardow (2021).",
)
production(dac_treatment, "direct air capture, recycling", "unit")
for input_activity, amount in [
    (concrete_treat, -3002000),
    (steel_recycling, -225024.75),
    (steel_disposal, -39710.25),
    (wool_treat, -2600),
    (pu_treat, -2600),
    (copper_treat, -2200),
    (aluminium_treat, -16000),
    (rubber_treat, -6300),
]:
    tech(dac_treatment, input_activity, amount)
remember(dac_treatment)

In [13]:
concrete = eidb.get(name="market for concrete, 30MPa", location="CH")
steel = eidb.get(name="market for reinforcing steel", location="GLO")
wool = eidb.get(name="market for stone wool", location="GLO")
chromium_steel = eidb.get(name="market for steel, chromium steel 18/8", location="GLO")
pu_foam = eidb.get(name="market for polyurethane, rigid foam", location="RER")
copper = eidb.get(name="market for copper, anode", location="GLO")
aluminium = eidb.get(name="market for aluminium, primary, ingot", location="IAI Area, Western and Central Europe")
paint = eidb.get(name="market for alkyd paint, white, without solvent, in 60% solution state", location="RER")
rubber = eidb.get(name="market for synthetic rubber", location="GLO")
aluminium_sheet = eidb.get(name="sheet rolling, aluminium", location="RER")
metal_working = eidb.get(name="metal working, average for copper product manufacturing", location="RER", product="metal working, average for copper product manufacturing")
copper_sheet = eidb.get(name="sheet rolling, copper", location="RER")
land_use = bsdb.get("fe9c3a98-a6d2-452d-a9a4-a13e64f1b95b")

dac_construction = recreate(
    "construction of direct air capture, 2016",
    name="construction of direct air capture, 2016",
    location="RER",
    unit="unit",
    comment="Construction of direct air capture infrastructure for a 4 kt CO2/a facility with 20 year lifetime.",
)
production(dac_construction, "direct air capture", "unit")
for input_activity, amount in [
    (concrete, 1501),
    (steel, 217590),
    (wool, 2600),
    (chromium_steel, 47145),
    (pu_foam, 2600),
    (copper, 2200),
    (aluminium, 16000),
    (paint, 1600),
    (rubber, 6300),
    (aluminium_sheet, 16000),
    (metal_working, 1600),
    (copper_sheet, 600),
    (dac_treatment, 1),
]:
    tech(dac_construction, input_activity, amount)
bio(dac_construction, land_use, 1045)
remember(dac_construction)

In [14]:
alumina = eidb.get(name="market for aluminium oxide, metallurgical", location="IAI Area, Western and Central Europe")
adsorbent = recreate(
    "adsorbent, amine on alumina",
    name="adsorbent, amine on alumina",
    location="DE",
    unit="kilogram",
    comment="Amine-on-alumina adsorbent based on Deutz and Bardow (2021) and Leonzio et al. (2022).",
)
production(adsorbent, "adsorbent, amine on alumina", "kilogram")
tech(adsorbent, pei, 0.557735849)
tech(adsorbent, alumina, 0.442264151)
remember(adsorbent)

## Created activities

In [15]:
created_activities = pd.DataFrame(created)
display(created_activities)

,activity name,code,database,location,unit,reference product,modelling status
0,"cooling energy production, at -30 °C, propylen...",cooling_minus_30_v1,disco2very,GLO,megajoule,"cooling energy, at -30 °C",created
1,"cooling energy production, at -75 °C",cooling_minus_75_v1,disco2very,GLO,megajoule,"cooling energy, at -75 °C",created
2,iridium production,iridium production,disco2very,GLO,kilogram,iridium,created
3,polyethyleneimine production,polyethyleneimine production,disco2very,DE,kilogram,polyethyleneimine,created
4,"treatment of direct air capture, 2016","treatment of direct air capture, 2016",disco2very,RER,unit,"direct air capture, recycling",created
5,"construction of direct air capture, 2016","construction of direct air capture, 2016",disco2very,RER,unit,direct air capture,created
6,"adsorbent, amine on alumina","adsorbent, amine on alumina",disco2very,DE,kilogram,"adsorbent, amine on alumina",created
